### Bienvenido a la Semana 6 Dia 3!

Vamos a experimentar con varios servidores MCP mas

In [ ]:
import os
import subprocess
from datetime import datetime

import httpx
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from IPython.display import Markdown, display

load_dotenv(override=True)

def get_ollama_base_url():
    """Usa OLLAMA_BASE_URL si existe; si no, detecta el host de Windows desde WSL."""
    configured_url = os.getenv("OLLAMA_BASE_URL") or os.getenv("OLLAMA_HOST")
    if configured_url:
        configured_url = configured_url.rstrip("/")
        return configured_url if configured_url.endswith("/v1") else f"{configured_url}/v1"

    route = subprocess.check_output(["ip", "route"], text=True)
    windows_host = next(line.split()[2] for line in route.splitlines() if line.startswith("default "))
    return f"http://{windows_host}:11434/v1"

ollama_url = get_ollama_base_url()
ollama_model_name = os.getenv("OLLAMA_MODEL", "qwen2.5:7b")

print("Usando Ollama en:", ollama_url)
print("Modelo Ollama:", ollama_model_name)

ollama_client = AsyncOpenAI(
    base_url=ollama_url,
    api_key="ollama",
    http_client=httpx.AsyncClient(
        timeout=120,
        trust_env=False
    )
)

ollama_model = OpenAIChatCompletionsModel(
    model=ollama_model_name,
    openai_client=ollama_client
)

### El primer tipo de servidor MCP: corre localmente, todo local

Aqui hay uno muy interesante: una memoria basada en grafo de conocimiento.

Es un almacen persistente de memoria con entidades, observaciones sobre ellas y relaciones entre ellas.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory


In [ ]:
params = {"command": "npx","args": ["-y", "mcp-memory-libsql"],"env": {"LIBSQL_URL": "file:./memory/ed.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
instructions = "Usas tus herramientas de entidades como memoria persistente para guardar y recordar informacion sobre tus conversaciones."
request = "Mi nombre es Ed. Soy ingeniero de LLMs. Estoy ensenando un curso sobre Agentes de IA, incluyendo el increible protocolo MCP. \
MCP es un protocolo para conectar agentes con herramientas, recursos y plantillas de prompts, y facilita integrar agentes de IA con distintas capacidades."
model = ollama_model

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=ollama_model, mcp_servers=[mcp_server])
    # con trace("conversacion"):
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=ollama_model, mcp_servers=[mcp_server])
    # con trace("conversacion"):
    result = await Runner.run(agent, "Mi nombre es Ed. Que sabes sobre mi?")
    display(Markdown(result.final_output))

### Nota sobre traces con Ollama:

Este lab ahora ejecuta el agente con Ollama localmente. Los enlaces de traces de OpenAI del lab original son opcionales y no hacen falta para la version con Ollama.

### El segundo tipo de servidor MCP: corre localmente y llama a un servicio web

### Brave Search - disculpas - esto necesita otra API key! Pero vuelve a ser gratis.

https://brave.com/search/api/

Configura tu cuenta y pon tu key en el archivo .env bajo `BRAVE_API_KEY`

In [ ]:
brave_api_key = os.getenv("BRAVE_API_KEY")
if not brave_api_key:
    raise ValueError("BRAVE_API_KEY no esta configurada. Agregala a tu .env antes de ejecutar las celdas de Brave Search.")

env = {"BRAVE_API_KEY": brave_api_key}
params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": env}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
instructions = "Puedes buscar informacion en la web y resumir brevemente los puntos clave."
request = f"Investiga las noticias mas recientes sobre el precio de las acciones de Amazon y resume brevemente su perspectiva. \
Como contexto, la fecha actual es {datetime.now().strftime('%Y-%m-%d')}"
model = ollama_model

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

### Nota sobre traces con Ollama:

Como esta version usa Ollama a traves de la API compatible con OpenAI, manten el flujo local e inspecciona directamente los outputs del notebook.

## Y ahora el tercer tipo: ejecucion remota

En realidad es bastante dificil encontrar un "servidor MCP remoto", tambien llamado "servidor MCP alojado" o "servidor MCP administrado".

No es un modelo comun para usar o compartir servidores MCP, y no existe una forma estandar de descubrir servidores MCP remotos.

Anthropic lista algunos servidores MCP remotos, pero son para aplicaciones pagas orientadas a usuarios empresariales:

https://docs.anthropic.com/en/docs/agents-and-tools/remote-mcp-servers

CloudFlare tiene herramientas para crear y desplegar tus propios servidores MCP remotos, pero esto no parece ser una practica muy comun:

https://developers.cloudflare.com/agents/guides/remote-mcp-server/


# Y volvemos al segundo tipo: el servidor MCP de Polygon.io

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">POR FAVOR LEE ESTO!!-</h2>
            <span style="color:#ff7800;">Este servicio de datos de mercados financieros tiene un plan GRATIS y un plan PAGO, y podemos usar cualquiera de los dos segun lo que prefieras.
            </span>
        </td>
    </tr>
</table>

## NUEVA SECCION: Introduccion a polygon.io

Polygon.io es un proveedor de datos financieros muy popular. Tiene un plan gratis y un plan pago. Y tambien tiene un servidor MCP!

Primero, lee sobre polygon.io en su excelente sitio web, incluyendo sus precios:

https://polygon.io

### Polygon.io Parte 1: servicio gratuito de Polygon.io (el plan pago sera totalmente opcional, por supuesto!)

1. Registrate en polygon.io (arriba a la derecha)  
2. Una vez que hayas iniciado sesion, selecciona "Keys" en la navegacion de la izquierda
3. Presiona el boton azul "New Key"
4. Copia el nombre de la key
5. Edita tu archivo .env y agrega la linea:

`POLYGON_API_KEY=xxxx`

In [19]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY no esta configurada")

In [20]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

PreviousCloseAgg(ticker='AAPL', close=300.23, high=303.2, low=296.52, open=297.9, timestamp=1778875200000, volume=54862836.0, vwap=300.4335)

### Envuelto en un modulo de python que cachea precios de cierre del dia

He creado un modulo de python `market.py` que usa esta API para consultar precios de acciones.

Pero la API gratuita tiene limites de uso bastante fuertes, asi que hice un pequeno truco: cuando pides el precio de una accion, esta funcion recupera todo el mercado accionario de cierre del dia y lo guarda en cache en nuestra base de datos.


In [21]:
from market import get_share_price
get_share_price("AAPL")

300.23

In [22]:
# sin preocupaciones por limites de uso!

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

300.23

### Y converti esto en un servidor MCP

Tal como hicimos con accounts.py; revisa `market_server.py`

In [23]:
params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

[Tool(name='lookup_share_price', description='This tool provides the current price of the given stock symbol.\n\n    Args:\n        symbol: the symbol of the stock\n    ', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'}, annotations=None)]

### Probemoslo!

Con suerte, el modelo de Ollama sera lo suficientemente listo para saber que el simbolo de Apple es AAPL

In [24]:
instructions = "Eres un asistente de mercado de valores. Cuando el usuario pregunte por el precio de una accion y tengas una herramienta disponible, debes usar la herramienta antes de responder. No adivines precios. Si conoces el ticker, usalo directamente. Responde de forma breve y clara en espanol."
request = "Cual es el precio de la accion de Apple?"
model = ollama_model

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

El precio actual de la acción de Apple (AAPL) es $300.23.

## Polygon.io Parte 2: Plan pago - totalmente opcional!

Si te interesa, puedes suscribirte al plan mensual para obtener datos de mercado mas actualizados y llamadas ilimitadas a la API.

Si decides hacerlo, tambien tiene sentido usar el servidor MCP completo que publico Polygon.io para aprovechar toda su funcionalidad.



In [25]:

params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@v0.1.0", "mcp_polygon"],
          "env": {"POLYGON_API_KEY": polygon_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools


[Tool(name='get_aggs', description='\n    List aggregate bars for a ticker over a given date range in custom time window sizes.\n    ', inputSchema={'properties': {'ticker': {'title': 'Ticker', 'type': 'string'}, 'multiplier': {'title': 'Multiplier', 'type': 'integer'}, 'timespan': {'title': 'Timespan', 'type': 'string'}, 'from_': {'anyOf': [{'type': 'string'}, {'type': 'integer'}, {'format': 'date-time', 'type': 'string'}, {'format': 'date', 'type': 'string'}], 'title': 'From'}, 'to': {'anyOf': [{'type': 'string'}, {'type': 'integer'}, {'format': 'date-time', 'type': 'string'}, {'format': 'date', 'type': 'string'}], 'title': 'To'}, 'adjusted': {'anyOf': [{'type': 'boolean'}, {'type': 'null'}], 'default': None, 'title': 'Adjusted'}, 'sort': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Sort'}, 'limit': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'title': 'Limit'}, 'params': {'anyOf': [{'additionalProperties': True, 'type': 'object'

### Vaya, son muchisimas herramientas!

Probemoslas; con suerte, la gran cantidad de herramientas no va a abrumar al modelo de Ollama!

Con el plan mensual de $29, no tenemos acceso a algunas APIs, asi que tuve que especificar que APIs se pueden llamar.

Si contrataste un plan mas grande, puedes quitar esa restriccion extra sin problema.

In [27]:
instructions = "Eres un asistente de mercado de valores. Debes usar las herramientas disponibles antes de responder precios de acciones. Para Apple usa el ticker AAPL. Si el usuario pide el precio mas reciente, llama a get_snapshot_ticker con AAPL. No inventes precios. Responde de forma breve y clara en espanol."
request = "Necesito saber el precio mas reciente de la accion de Apple. Apple cotiza con el ticker AAPL. Usa obligatoriamente la herramienta get_snapshot_ticker con el ticker AAPL. Despues de usar la herramienta, responde en una frase corta indicando el precio encontrado y aclarando que corresponde a Apple/AAPL. No inventes el precio si la herramienta falla."
model = ollama_model

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Podría haber ocurrido un error en la solicitud. Utilizando la herramienta get_snapshot_ticker con el ticker AAPL y especificando el market_type, no se obtuvo una respuesta válida. Por favor intenta de nuevo más tarde o prueba con otra opción disponible.

## Configuracion de tu archivo .env

Si decides usar un plan pago, agrega esto a tu archivo .env para indicarlo:

`POLYGON_PLAN=paid`

Y si decides ir hasta el plan con API en tiempo real, entonces agrega:

`POLYGON_PLAN=realtime`

In [ ]:
load_dotenv(override=True)

polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

if is_paid_polygon:
    print("Elegiste suscribirte al plan pago de Polygon, asi que el codigo buscara precios con 15 minutos de retraso")
elif is_realtime_polygon:
    print("Vaya - elegiste suscribirte al plan en tiempo real de Polygon, asi que el codigo buscara precios en tiempo real")
else:
    print("Segun tu archivo .env, elegiste usar el plan gratuito de Polygon, asi que el codigo buscara precios de cierre del dia")

## Y eso es todo por hoy!

Quite la parte de este lab que usa el servidor MCP de "Financial Datasets", porque es inferior: mas caro y con menos APIs.

Asi podemos usar el mismo proveedor tanto para APIs gratuitas como pagas.

Pero si quieres ver ese codigo, puedes revisar una version anterior en el historial de git.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ejercicios</h2>
            <span style="color:#ff7800;">Explora marketplaces de servidores MCP e integra uno propio usando los 3 enfoques.
            </span>
        </td>
    </tr>
</table>